In [ ]:
import numpy as np
import cv2
from pathlib import Path
import torch
import h5py
import re

In [101]:
homography_path = "week_32/box_3/2024_08_06_16_52_08_recordings/homography.npz"
labelsdir = Path("week_32/box_3/2024_08_06_16_52_08_recordings/labels")
event_path = "week_32/box_3/2024_08_06_16_52_08_recordings/event_frames/clip_073_frames.h5"

In [78]:
data = np.load(homography_path)
H = data["H"].astype(np.float32)
W1 = data["W1"]
H1 = data["H1"]
W2 = data["W2"]
H2 = data["H2"]
N = 5000 # number of event frames to display

In [46]:
FRAMES_PER_CLIP = 5400

In [41]:
# YOLO labels mapping
yolo_labels = {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microwave', 69: 'oven', 70: 'toaster', 71: 'sink', 72: 'refrigerator', 73: 'book', 74: 'clock', 75: 'vase', 76: 'scissors', 77: 'teddy bear', 78: 'hair drier', 79: 'toothbrush'}

In [102]:
with h5py.File(event_path, "r") as f:
    frames = f["frames"][:]

##### Display test video of transferred labels

In [82]:
for i in range(N):
    t = frames[i]

    clip_idx = (i + FRAMES_PER_CLIP*2) // FRAMES_PER_CLIP
    frame_in_clip = i % FRAMES_PER_CLIP

    img = (t > 0).astype(np.uint8) * 255
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    label_frame = labelsdir / f"clip_{clip_idx:03d}/{frame_in_clip}.txt"
    
    if label_frame.exists():
        for s in label_frame.read_text().splitlines():
            cls, x, y, w, h = s.split()[:5]
            x, y, w, h = map(float, (x, y, w, h))

            x*=W2; y*=H2; w*=W2; h*=H2
            c = np.float32([[x-w/2,y-h/2],[x+w/2,y-h/2],[x+w/2,y+h/2],[x-w/2,y+h/2]]).reshape(-1,1,2)

            # warp to event frame
            c = cv2.perspectiveTransform(c, H).reshape(-1,2)
            x1,y1 = c.min(0); x2,y2 = c.max(0)
            x1,y1 = max(0,int(x1)), max(0,int(y1))
            x2,y2 = min(W1-1,int(x2)), min(H1-1,int(y2))

            if x2 > x1 and y2 > y1:
                cv2.rectangle(img, (x1,y1), (x2,y2), (0,0,255), 2)
                cv2.putText(img, yolo_labels[int(cls)], (x1, max(0,y1-5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

    cv2.imshow("Event frame with warped YOLO labels", img)
    if cv2.waitKey(30) & 0xFF == ord('q'):   # ~33 fps; change as you like
        break

cv2.destroyAllWindows()

##### Save warped YOLO labels for event frames

In [85]:
out_labels_dir = Path("week_32/box_3/2024_08_06_16_52_08_recordings/event_labels")
out_labels_dir.mkdir(parents=True, exist_ok=True)

In [97]:
vlen_f32 = h5py.vlen_dtype(np.dtype("float32"))

def numeric_txt_key(p: Path):
    return int(p.stem)

def warp_label_file(label_file: Path, H, W2, H2, W1, H1):
    rows = []

    if not label_file.exists():
        return np.empty((0, 6), dtype=np.float32)

    for s in label_file.read_text().splitlines():
        parts = s.split()
        if len(parts) < 6:
            continue

        cls, x, y, w, h, conf = parts[:6]
        cls = float(cls)
        x, y, w, h, conf = map(float, (x, y, w, h, conf))

        # YOLO normalized -> pixel coords in source frame
        x *= W2
        y *= H2
        w *= W2
        h *= H2

        corners = np.float32([
            [x - w / 2, y - h / 2],
            [x + w / 2, y - h / 2],
            [x + w / 2, y + h / 2],
            [x - w / 2, y + h / 2]
        ]).reshape(-1, 1, 2)

        # Warp to event frame
        warped = cv2.perspectiveTransform(corners, H).reshape(-1, 2)
        x1, y1 = warped.min(axis=0)
        x2, y2 = warped.max(axis=0)

        # Clip to image bounds
        x1, y1 = max(0.0, x1), max(0.0, y1)
        x2, y2 = min(W1 - 1.0, x2), min(H1 - 1.0, y2)

        if x2 <= x1 or y2 <= y1:
            continue

        # Back to YOLO normalized in event frame
        xn = (x1 + x2) / 2 / W1
        yn = (y1 + y2) / 2 / H1
        wn = (x2 - x1) / W1
        hn = (y2 - y1) / H1

        rows.append([cls, xn, yn, wn, hn, conf])

    return np.asarray(rows, dtype=np.float32).reshape(-1, 6)

# Automatically find all clip folders
clip_dirs = sorted(
    [p for p in labelsdir.iterdir() if p.is_dir() and p.name.startswith("clip_")],
    key=lambda p: int(p.name.split("_")[-1])
)

for clip_dir in clip_dirs:
    out_h5 = out_labels_dir / f"{clip_dir.name}.h5"
    N = 5400

    with h5py.File(out_h5, "w") as f:
        dset = f.create_dataset("labels", shape=(N,), dtype=vlen_f32)
        f.attrs["format"] = "per-frame vlen flattened (Ni*6,), reshape to (-1,6)"
        f.attrs["columns"] = "[cls, x, y, w, h, conf] (YOLO-normalized to event frame)"

        for frame_idx in range(N):
            label_file = clip_dir / f"{frame_idx}.txt"
            arr = warp_label_file(label_file, H, W2, H2, W1, H1)
            dset[frame_idx] = arr.reshape(-1)

    print(f"Saved: {out_h5}")

print("Finished.")

Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_000.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_001.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_002.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_003.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_004.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_005.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_006.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_007.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_008.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_009.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_010.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_011.h5
Saved: week_32\box_3\2024_08_06_16_52_08_recordings\event_labels\clip_012.h5

In [103]:
FRAMES_PER_CLIP = 5400

event_labels_dir = Path("week_32/box_3/2024_08_06_16_52_08_recordings/event_labels")

open_h5 = None
current_clip_idx = None
labels_dset = None

for i in range(N):
    t = frames[i]

    clip_idx = (i+5400*73) // FRAMES_PER_CLIP
    frame_in_clip = i % FRAMES_PER_CLIP

    # Open the correct clip file only when clip changes
    if current_clip_idx != clip_idx:
        if open_h5 is not None:
            open_h5.close()

        h5_path = event_labels_dir / f"clip_{clip_idx:03d}.h5"
        if not h5_path.exists():
            print(f"Missing label file: {h5_path}")
            open_h5 = None
            labels_dset = None
        else:
            open_h5 = h5py.File(h5_path, "r")
            labels_dset = open_h5["labels"]

        current_clip_idx = clip_idx

    img = (t > 0).astype(np.uint8) * 255
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    if labels_dset is not None and frame_in_clip < len(labels_dset):
        flat = labels_dset[frame_in_clip]   # shape: (Ni*5,)
        arr = np.asarray(flat, dtype=np.float32).reshape(-1, 6)

        for cls, xn, yn, wn, hn, conf in arr:
            # YOLO normalized in event frame -> pixel box
            x = xn * W1
            y = yn * H1
            w = wn * W1
            h = hn * H1

            x1 = max(0, int(x - w / 2))
            y1 = max(0, int(y - h / 2))
            x2 = min(W1 - 1, int(x + w / 2))
            y2 = min(H1 - 1, int(y + h / 2))

            if x2 > x1 and y2 > y1:
                cls = int(cls)
                name = yolo_labels[cls] if 0 <= cls < len(yolo_labels) else str(cls)
                label = f"{yolo_labels[int(cls)]} {conf:.2f}"

                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(
                    img,
                    label, #name
                    (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 0, 255),
                    2
                )

    cv2.imshow("Event frame with warped YOLO labels", img)
    if cv2.waitKey(30) & 0xFF == ord("q"):
        break

if open_h5 is not None:
    open_h5.close()

cv2.destroyAllWindows()